# 05 – Data Transformation

Transform raw supplier data into a normalized, platform-independent product
dataset that can be used by later validation and ecommerce import steps.

## Responsibility

This notebook:

- loads the supplier price list and product feed
- validates that required source columns are available
- identifies price levels within product models
- merges commercial and product information
- normalizes product and variant fields into a consistent schema
- applies documented supplier-specific corrections
- extracts and normalizes product/variant image references
- validates the transformed dataset before export

## Inputs

- Supplier price list
- Supplier product feed

## Outputs

- `normalized_products.csv`
- `valid_products.csv`
- `data_issues.csv`
- `normalized_product_images.csv`
- `valid_product_images.csv`

The output is intentionally platform-independent. Shopify-specific transformation
and synchronization belong in later notebooks.

## Setup

In [ ]:
from pathlib import Path
import pandas as pd
import xml.etree.ElementTree as ET


SUPPLIER = "snickers"
SUPPLIER_NAME = "Snickers"
IMAGE_SOURCE_COLUMN = "This_x0020_is_x0020_the_x0020_main_x0020_image"

DATA_DIR = Path("../data") / SUPPLIER

PRICE_LIST_PATH = (
    DATA_DIR
    / "price_list"
    / "Prislista_Snickers_WW_202609.xlsx"
)

PRODUCT_FEED_PATH = (
    DATA_DIR
    / "product_feeds"
    / "PP_Export_Snickers_018_sv.xml"
)

In [ ]:
for source_path in (PRICE_LIST_PATH, PRODUCT_FEED_PATH):
    if not source_path.exists():
        raise FileNotFoundError(f"Source file not found: {source_path}")

print("Source files found.")

## Load Source Data

In [ ]:
price_list = pd.read_excel(
    PRICE_LIST_PATH,
    header=1,
)

tree = ET.parse(PRODUCT_FEED_PATH)
root = tree.getroot()

## Prepare Source Data

In [ ]:
product_records = []

for product in root.findall(".//ProductInfo"):
    record = {
        child.tag: child.text
        for child in product
    }
    product_records.append(record)

product_feed = pd.DataFrame(product_records)

print(f"Product Feed rows: {len(product_feed):,}")
print(f"Product Feed columns: {len(product_feed.columns)}")

product_feed.head()

## Validate Required Fields

In [ ]:
required_price_list_columns = {
    "Artikelnr",
    "Modell",
    "Färg",
    "Storlekskod",
    "Nettopris",
    "RRP Pris",
    "EAN-nr. styck",
    "Gäller från",
}

required_product_feed_columns = {
    "StockCode",
    "Name",
    "Intro",
    "Feature1",
    "Feature2",
    "Feature3",
    "Feature4",
    "Feature5",
    "Feature6",
    "Feature7",
    "TechnicalDescription",
    "EAN_text",
    IMAGE_SOURCE_COLUMN,
}

missing_price_list_columns = (
    required_price_list_columns - set(price_list.columns)
)

missing_product_feed_columns = (
    required_product_feed_columns - set(product_feed.columns)
)

assert not missing_price_list_columns, (
    f"Missing Price List columns: {sorted(missing_price_list_columns)}"
)

assert not missing_product_feed_columns, (
    f"Missing Product Feed columns: {sorted(missing_product_feed_columns)}"
)

print("All required source columns are available.")

## Identify Price Levels Within Models

In [ ]:
price_list_prepared = price_list.copy()

price_list_prepared["Modell"] = (
    price_list_prepared["Modell"]
    .astype("string")
    .str.strip()
)

price_list_prepared["Artikelnr"] = (
    price_list_prepared["Artikelnr"]
    .astype("string")
    .str.strip()
)

price_list_prepared["Storlekskod"] = (
    price_list_prepared["Storlekskod"]
    .astype("string")
    .str.strip()
)

price_list_prepared["Färg"] = (
    price_list_prepared["Färg"]
    .astype("string")
    .str.strip()
)

price_list_prepared["Nettopris"] = pd.to_numeric(
    price_list_prepared["Nettopris"],
    errors="coerce",
)

price_list_prepared["RRP Pris"] = pd.to_numeric(
    price_list_prepared["RRP Pris"],
    errors="coerce",
)

price_list_prepared.head()

Identify variants whose price differs from the model's base price.
These variants are excluded from the standard webshop import.

In [ ]:
price_list_prepared["Model Min Net Price"] = (
    price_list_prepared
    .groupby("Modell")["Nettopris"]
    .transform("min")
)

price_list_prepared["Model Min RRP Price"] = (
    price_list_prepared
    .groupby("Modell")["RRP Pris"]
    .transform("min")
)

price_list_prepared["Special Priced"] = (
    (price_list_prepared["Nettopris"] > price_list_prepared["Model Min Net Price"])
    | (price_list_prepared["RRP Pris"] > price_list_prepared["Model Min RRP Price"])
)

price_list_prepared[
    [
        "Artikelnr",
        "Modell",
        "Färg",
        "Storlekskod",
        "Nettopris",
        "Model Min Net Price",
        "RRP Pris",
        "Model Min RRP Price",
        "Special Priced",
    ]
].head(20)


## Exclude Special-Priced Variants

In [ ]:
webshop_price_list = (
    price_list_prepared.loc[
        ~price_list_prepared["Special Priced"]
    ]
    .copy()
    .reset_index(drop=True)
)

print(f"Rows before filtering: {len(price_list_prepared):,}")
print(f"Special-priced rows excluded: {price_list_prepared['Special Priced'].sum():,}")
print(f"Rows remaining: {len(webshop_price_list):,}")

webshop_price_list.head()

In [ ]:
model_price_summary = (
    price_list_prepared
    .groupby("Modell")
    .agg(
        rows=("Artikelnr", "size"),
        net_price_levels=("Nettopris", "nunique"),
        rrp_price_levels=("RRP Pris", "nunique"),
        min_net_price=("Nettopris", "min"),
        max_net_price=("Nettopris", "max"),
        min_rrp_price=("RRP Pris", "min"),
        max_rrp_price=("RRP Pris", "max"),
        special_priced_rows=("Special Priced", "sum"),
    )
    .reset_index()
)

model_price_summary[
    model_price_summary["special_priced_rows"] > 0
].sort_values(
    "special_priced_rows",
    ascending=False,
).head(30)

In [ ]:
print(
    "Models with multiple Net Price levels:",
    (model_price_summary["net_price_levels"] > 1).sum(),
)

print(
    "Models with multiple RRP Price levels:",
    (model_price_summary["rrp_price_levels"] > 1).sum(),
)

print(
    "Models containing excluded variants:",
    (model_price_summary["special_priced_rows"] > 0).sum(),
)

In [ ]:
excluded_special_variants = (
    price_list_prepared.loc[
        price_list_prepared["Special Priced"]
    ]
    .copy()
)

excluded_special_variants.head()

#### Observed

- Price levels are compared within each supplier model.
- Variants priced above the lowest Net Price or RRP Price within the same model are classified as special-priced.
- The same pricing pattern was verified for representative trouser and jacket models.
- Special-priced variants are excluded before the Price List is merged with the Product Feed.
- The rule identifies higher-priced variants without relying on specific size suffixes or numeric size codes.

## Merge Price List and Product Feed

In [ ]:
price_articles = set(webshop_price_list["Artikelnr"])

feed_articles = set(product_feed["StockCode"])

print(f"Webshop-eligible Price List articles: {len(price_articles):,}")
print(f"Product Feed articles: {len(feed_articles):,}")

print(f"Articles in both sources: {len(price_articles & feed_articles):,}")
print(f"Only in Price List: {len(price_articles - feed_articles):,}")
print(f"only in Product Feed: {len(feed_articles - price_articles):,}")

In [ ]:
missing_from_product_feed = webshop_price_list.loc[
    ~webshop_price_list["Artikelnr"].isin(product_feed["StockCode"])
].copy()

missing_from_product_feed[
    [
        "Artikelnr",
        "Modell",
        "Färg",
        "Storlekskod",
        "Gäller från",
    ]
].head(50)

#### Observed

- Only webshop-eligible articles available in both the Price List and Product Feed can be fully enriched.
- The Price List is treated as the authoritative source for currently active and priced variants.
- Articles that exist in the Price List but are missing from the Product Feed are reported for review.
- Product Feed variants that are not present in the Price List are excluded from the webshop dataset.
- This prevents discontinued or non-priced variants from being imported even if descriptive data still exists in the Product Feed.
- Runtime counts are printed by the code above instead of being hardcoded in the documentation.

In [ ]:
merged_products = webshop_price_list.merge(
    product_feed,
    how="inner",
    left_on="Artikelnr",
    right_on="StockCode",
    validate="one_to_one",
    suffixes=("_price", "_feed"),
)

print(f"Rows in webshop Price List: {len(webshop_price_list):,}")
print(f"Rows after inner merge: {len(merged_products):,}")
print(f"Unique articles after merge: {merged_products['Artikelnr'].nunique():,}")

merged_products.head()

#### Observed:

- An inner join is performed between the filtered Price List and the Product Feed.
- Only articles present in both sources are retained.
- The merge preserves one row per supplier article number.
- The merged dataset forms the basis for the subsequent ecommerce transformation.

## Create Normalized Product Structure

In [ ]:
normalized_columns = [
    "supplier",
    "product_id",
    "product_name",
    "introduction_text",
    "description",
    "variant_sku",
    "color",
    "size",
    "price_sek",
    "image_url",
    "ean",
]

normalized_products = pd.DataFrame(
    index=merged_products.index,
    columns=normalized_columns,
)

normalized_products["supplier"] = SUPPLIER_NAME

#### Observed:

- The target import structure is created on the mapping strategy defined in notebook 04.
- Standard product fields are combined with the verified customer-choice fields required for supplier variants.
- The dataframe will be populated from the merged supplier dataset in the following transformation steps.

## Populate Product Information

In [ ]:
normalized_products["product_id"] = (
    merged_products["Modell"]
    .astype("string")
    .str.strip()
)

normalized_products["product_name"] = merged_products["Name"]

normalized_products["introduction_text"] = merged_products["Intro"]

description_columns = [
    "Feature1",
    "Feature2",
    "Feature3",
    "Feature4",
    "Feature5",
    "Feature6",
    "Feature7",
    "TechnicalDescription",
]

normalized_products["description"] = (
    merged_products[description_columns]
    .fillna("")
    .astype(str)
    .apply(
        lambda row: "\n\n".join(
            text.strip()
            for text in row
            if text.strip()
        ),
        axis=1,
    )
)
normalized_products[
    [
        "product_id",
        "product_name",
        "introduction_text",
        "description",
    ]
].head()

#### Observed:

- Product-level information is populated from the merged supplier dataset.
- Product names, introductory text and descriptions originate from the Product Feed.
- The product description is generated by combining the available feature fields and the technical description into a single text.

## Populate Variant Information

In [ ]:
normalized_products["variant_sku"] = (
    merged_products["Artikelnr"]
    .astype("string")
    .str.strip()
    .to_numpy()
)

normalized_products["color"] = (
    merged_products["Färg"].to_numpy()
)

normalized_products["size"] = (
    merged_products["Storlekskod"]
    .astype("string")
    .str.strip()
    .replace({
        "012": "6XL",
        "No size": "One size",
    })
    .to_numpy()
)

normalized_products["price_sek"] = (
    merged_products["RRP Pris"].to_numpy()
)

normalized_products["ean"] = (
    merged_products["EAN_text"]
    .fillna(merged_products["EAN-nr. styck"])
    .to_numpy()
)

normalized_products[
    ["product_id", "variant_sku", "color", "size", "price_sek", "ean"]
].head()

### Supplier-specific data corrections

Known exceptions in the supplier feed are corrected here before the normalized
dataset is exported.

For Snickers, product `5499` is a one-size product but has no size value in the
source data. The missing value is therefore normalized to `One size`.

Keep supplier-specific corrections isolated in this section so the main
transformation logic remains reusable for other suppliers.

In [ ]:
normalized_products.loc[
    normalized_products["product_id"].eq("5499")
    & normalized_products["size"].isna(),
    "size",
] = "One size"

#### Observed:

- Variant-specific information is populated for each supplier article.
- The supplier article number is used as the variant SKU.
- Colour and size are transferred directly from the filtered Price List.
- Retail prices are imported from the supplier Price List.
- EAN values are primarily taken from the Product Feed, with the Price List used as a fallback when necessary.
- Internal supplier size codes are normalized where required (e.g. "012" → "6XL").

## Populate Media

In [ ]:
source_image_column = merged_products[IMAGE_SOURCE_COLUMN]

normalized_products["image_url"] = (
    source_image_column
    .astype("string")
    .str.split(",")
    .str[0]
    .str.strip()
    .to_numpy()
)

normalized_products[
    ["product_id", "variant_sku", "image_url"]
].head()

In [ ]:
normalized_products.isna().sum()

The first supplier image is used as the normalized primary product image.

All supplier image URLs are also expanded into a separate image table so later
steps can process the complete product and variant image gallery independently.

In [ ]:
normalized_product_images = (
    merged_products[
        [
            "Modell",
            "Artikelnr",
            "Färg",
            "This_x0020_is_x0020_the_x0020_main_x0020_image",
        ]
    ]
    .rename(
        columns={
            "Modell": "product_id",
            "Artikelnr": "variant_sku",
            "Färg": "color",
        }
    )
    .assign(
        image_url=lambda df:
            df["This_x0020_is_x0020_the_x0020_main_x0020_image"]
            .astype("string")
            .str.split(",")
    )
    .explode("image_url")
)

normalized_product_images["image_url"] = (
    normalized_product_images["image_url"].str.strip()
)

normalized_product_images = (
    normalized_product_images[
        normalized_product_images["image_url"].notna()
        & normalized_product_images["image_url"].ne("")
    ]
    .drop(
        columns=["This_x0020_is_x0020_the_x0020_main_x0020_image"]
    )
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"Image rows: {len(normalized_product_images):,}")
print(f"Unique images: {normalized_product_images['image_url'].nunique():,}")

normalized_product_images.head()

## Validate Normalized Output

Inspect the normalized dataset before export and separate valid products from
models that require review.

A model is held for review if any variant is missing a required field such as
product name, color or image. This prevents incomplete product models from
continuing to later import steps.

In [ ]:
normalized_products.info()

In [ ]:
normalized_products.head()

In [ ]:
duplicate_skus = normalized_products["variant_sku"].duplicated().sum()

assert duplicate_skus == 0, (
    f"Duplicate variant SKUs found: {duplicate_skus}"
)

print("Variant SKU validation passed.")

In [ ]:
required_fields = [
    "product_name",
    "color",
    "size",
    "price_sek",
    "image_url",
]

assert normalized_products["product_id"].notna().all(), (
    "Missing product_id values found."
)

assert normalized_products["variant_sku"].notna().all(), (
    "Missing variant_sku values found."
)

missing_required_mask = (
    normalized_products[required_fields].isna()
    | normalized_products[required_fields]
        .astype("string")
        .apply(lambda col: col.str.strip().eq(""))
)

invalid_model_ids = normalized_products.loc[
    missing_required_mask.any(axis=1),
    "product_id",
].unique()

valid_products = normalized_products.loc[
    ~normalized_products["product_id"].isin(invalid_model_ids)
].copy()

data_issues = normalized_products.loc[
    normalized_products["product_id"].isin(invalid_model_ids)
].copy()

print(f"Valid variants: {len(valid_products):,}")
print(f"Variants held for review: {len(data_issues):,}")
print(f"Models held for review: {data_issues['product_id'].nunique():,}")

In [ ]:
issue_flags = pd.DataFrame({
    "product_id": normalized_products["product_id"],
    "missing_product_name": missing_required_mask["product_name"],
    "missing_color": missing_required_mask["color"],
    "missing_image": missing_required_mask["image_url"],
    "missing_size": missing_required_mask["size"],
    "missing_price": missing_required_mask["price_sek"],
})

issue_summary = (
    issue_flags[
        issue_flags["product_id"].isin(invalid_model_ids)
    ]
    .groupby("product_id")
    .agg(
        missing_product_name=("missing_product_name", "any"),
        missing_color=("missing_color", "any"),
        missing_image=("missing_image", "any"),
        missing_size=("missing_size", "any"),
        missing_price=("missing_price", "any"),
    )
    .reset_index()
)

data_issues = data_issues.merge(
    issue_summary,
    on="product_id",
    how="left",
)

In [ ]:
issue_columns = [
    "missing_product_name",
    "missing_color",
    "missing_image",
    "missing_size",
    "missing_price",
]

data_issues["issue_reason"] = data_issues[issue_columns].apply(
    lambda row: ", ".join(
        column.replace("missing_", "")
        for column in issue_columns
        if row[column]
    ),
    axis=1,
)

data_issues[
    ["product_id", "product_name", "issue_reason"]
].drop_duplicates().sort_values("product_id")

In [ ]:
model_issues = (
    data_issues
    .groupby("product_id", as_index=False)
    .agg(
        product_name=("product_name", "first"),
        issue_reason=("issue_reason", "first"),
        affected_variants=("variant_sku", "size"),
    )
    .sort_values("product_id")
)

model_issues

## Export Normalized Product Data

In [ ]:
valid_output_path = DATA_DIR / "valid_products.csv"
issues_output_path = DATA_DIR / "data_issues.csv"

valid_products.to_csv(
    valid_output_path,
    index=False,
    encoding="utf-8-sig",
)

data_issues.to_csv(
    issues_output_path,
    index=False,
    encoding="utf-8-sig",
)

print(f"Exported valid products: {valid_output_path}")
print(f"Exported data issues: {issues_output_path}")

In [ ]:
output_path = DATA_DIR / "normalized_products.csv"

normalized_products.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig",
)

print(f"Exported: {output_path}")

In [ ]:
normalized_product_images.to_csv(
    DATA_DIR / "normalized_product_images.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Exported normalized_product_images.csv")

In [ ]:
valid_product_images = normalized_product_images[
    normalized_product_images["product_id"].isin(
        valid_products["product_id"]
    )
].copy()

print(f"Valid image rows: {len(valid_product_images):,}")
print(
    f"Valid unique images: "
    f"{valid_product_images['image_url'].nunique():,}"
)
print(
    f"Valid models with images: "
    f"{valid_product_images['product_id'].nunique():,}"
)

In [ ]:
valid_product_images.to_csv(
    DATA_DIR / "valid_product_images.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Exported valid_product_images.csv")